In [ ]:
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import locale

# Intentar establecer el idioma a español para las fechas
try:
    locale.setlocale(locale.LC_TIME, 'es_ES.UTF-8')
except:
    try:
        locale.setlocale(locale.LC_TIME, 'spanish')
    except:
        pass # Si falla, usará el idioma por defecto pero el formato será correcto

# 1. Configuración de Identidad Visual
PGA_COLORS = {
    'pumpkin': '#ED7D31',
    'yellow': '#F9C035',
    'gray': '#595959',
    'platinum': '#D3D4D9',
    'white': '#FFFFFF'
}

CSS = f"""
<style>
    .pga-header {{
        font-family: 'Lato', sans-serif;
        color: {PGA_COLORS['gray']};
        border-left: 5px solid {PGA_COLORS['pumpkin']};
        padding-left: 15px;
        margin-top: 30px;
        margin-bottom: 15px;
    }}
    .widget-label {{ font-weight: bold; color: {PGA_COLORS['gray']}; }}
</style>
"""
display(HTML(CSS))


# Mapeo de colores para las líneas del gráfico
LINE_COLORS = {
   'Inflacion BCV': '#D3D4D9',
    'Inf. Acum BCV': '#ED7D31',
    'Inflación Acumulada': '#ED7D31',
    'Tasa bcv': '#F9C035',
    'Deval. Acum BCV': '#595959',
    'Devaluación Acumulada BCV': '#F9C035',
    'Devaluación Acumulada USDT': '#595959'
}

# 2. Carga y preparación de la hoja 'Data'
file_path = 'Indicadores_abril_2026.xlsx'
df = pd.read_excel(file_path, sheet_name='Data')
df.columns = df.columns.str.strip()

# Filtro 2025+ y limpieza de Año
df_2025 = df[df['Año'] >= 2025].copy()
df_2025['Año'] = df_2025['Año'].astype(int)

meses_dict = {
    'ENERO': 1, 'FEBRERO': 2, 'MARZO': 3, 'ABRIL': 4, 'MAYO': 5, 'JUNIO': 6,
    'JULIO': 7, 'AGOSTO': 8, 'SEPTIEMBRE': 9, 'OCTUBRE': 10, 'NOVIEMBRE': 11, 'DICIEMBRE': 12
}

df_2025['Mes_Num'] = df_2025['Mes'].str.strip().str.upper().map(meses_dict)
df_2025['Fecha_DT'] = pd.to_datetime(
    df_2025['Año'].astype(str) + '-' + 
    df_2025['Mes_Num'].astype(int).astype(str).str.zfill(2) + '-01'
)
df_2025 = df_2025.sort_values('Fecha_DT')

# --- PREPARACIÓN DE DATOS ---
# 1. Limpiamos espacios en blanco ocultos en los nombres de las columnas
df_2025.columns = df_2025.columns.str.strip()
df_2025['Año'] = df_2025['Año'].astype(int).astype(str)

# --- FUNCIÓN DE GRÁFICO MAESTRA ---
def plot_pga_master(data, columns, title):
    if not columns: return
    fig, ax = plt.subplots(figsize=(11, 4))
    for col in columns:
        color = LINE_COLORS.get(col, '#D3D4D9')
        ax.plot(data['Fecha_DT'], data[col], marker='o', label=col, linewidth=2.5, color=color)
    
    ax.set_title(title, fontsize=12, color=PGA_COLORS['gray'], fontweight='bold')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b-%Y'))
    ax.xaxis.set_major_locator(mdates.MonthLocator())
    plt.xticks(rotation=45, color=PGA_COLORS['gray'])
    
    if any(c in col for c in ['Inflación', 'Devaluación', 'Acum', 'Acumulada', 'Inflacion']):
        ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{x:.1%}'))
    else:
        ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'Bs. {x:,.2f}'))
    
    ax.grid(axis='y', linestyle='--', alpha=0.3)
    ax.legend(frameon=False, loc='upper left')
    plt.tight_layout()
    plt.show()

# --- CLASE DE SELECTORES ---
class IconSelector:
    def __init__(self, options_dict, callback):
        self.buttons = {}
        for key, (icon, label) in options_dict.items():
            btn = widgets.ToggleButton(
                value=False, description=label, icon=icon,
                layout=widgets.Layout(width='200px', margin='5px'),
                style={'font_weight': 'bold'}
            )
            btn.observe(self._handle_click, names='value')
            btn.observe(callback, names='value')
            self.buttons[key] = btn
            
    def _handle_click(self, change):
        change['owner'].button_style = 'warning' if change['new'] else ''

    @property
    def selected(self):
        return [k for k, v in self.buttons.items() if v.value]

    def get_widget(self):
        return widgets.HBox(list(self.buttons.values()))

# --- CONFIGURACIÓN DE NOMBRES (Solicitud de Salida) ---

# Parte 1: Usamos "Acum" y la Tasa Promedio (Columna T)
opciones_h = {
    'Inflacion BCV': ('percent', ' Inflación Mes'),
    'Inf. Acum BCV': ('line-chart', ' Inf. Acum BCV'),
    
    'Deval. Acum BCV': ('bank', ' Deval. Acum BCV'),
    'Tasa Promedio BCV': ('calculator', ' Promedio Tasa') # <--- Apuntando estrictamente a la Columna T
}

# Parte 2: Usamos "Acumulada" (Mantiene la tasa de cierre para la devaluación)
opciones_s = {
    'Inflación Acumulada': ('line-chart', ' Inflación'),
    'Devaluación Acumulada BCV': ('university', ' Deval. BCV'),
    'Devaluación Acumulada USDT': ('dollar', ' Deval. USDT')
}

out1, out2 = widgets.Output(), widgets.Output()

def actualizar_h(*args):
    with out1:
        clear_output(wait=True)
        sel = selector_h.selected
        if not sel: return
        
        temp_df = df_2025.copy()
        temp_df['Inf. Acum BCV'] = temp_df['Inflación acumulada BCV']
        temp_df['Deval. Acum BCV'] = temp_df['Devaluación acumulada BCV']
        
        fmt = {c: ('{:.2%}' if 'Tasa' not in c else 'Bs. {:,.2f}') for c in sel}
        display(temp_df[['Año', 'Mes'] + sel].style.hide(axis='index').format(fmt))
        plot_pga_master(temp_df, sel, "Histórico de Indicadores (Métrica Acum)")

def actualizar_s(*args):
    with out2:
        clear_output(wait=True)
        sel, m_base = selector_s.selected, drop_mes.value
        if not sel: return
        
        f_base = pd.to_datetime(m_base)
        df_d = df_2025[df_2025['Fecha_DT'] >= f_base].copy()
        prev = df_2025[df_2025['Fecha_DT'] < f_base].tail(1)
        
        # Cálculos de acumulación
        df_d['Inflación Acumulada'] = (1 + df_d['Inflacion BCV']).cumprod() - 1
        
        # Matemáticas de devaluación con la última tasa del mes
        t_ref = prev['Tasa bcv'].values[0] if not prev.empty else df_d['Tasa bcv'].iloc[0]/(1 + df_d['Inflacion BCV'].iloc[0])
        df_d['Devaluación Acumulada BCV'] = (df_d['Tasa bcv'] / t_ref) - 1
        
        if 'Tasa Promedio USDT' in df_2025.columns:
            tu_ref = prev['Tasa Promedio USDT'].values[0] if not prev.empty else df_d['Tasa Promedio USDT'].iloc[0] / 1.02
            df_d['Devaluación Acumulada USDT'] = (df_d['Tasa Promedio USDT'] / tu_ref) - 1
        else:
            df_d['Devaluación Acumulada USDT'] = 0

        display(df_d[['Año', 'Mes'] + sel].style.hide(axis='index').format({c: '{:.2%}' for c in sel}))
        plot_pga_master(df_d, sel, f"Simulador: Comportamiento Acumulada desde {m_base}")

# --- RENDER ---
selector_h = IconSelector(opciones_h, actualizar_h)
selector_s = IconSelector(opciones_s, actualizar_s)
drop_mes = widgets.Dropdown(options=df_2025['Fecha_DT'].dt.strftime('%Y-%m').unique(), description='📅 Mes Base:')
drop_mes.observe(actualizar_s, names='value')

display(HTML(f"<div style='border-left: 5px solid {PGA_COLORS['pumpkin']}; padding-left:15px; margin-top:20px;'><h2>1. Histórico (Métricas Acum)</h2></div>"))
display(widgets.VBox([selector_h.get_widget(), out1]))

display(HTML(f"<hr style='margin:40px 0;'><div style='border-left: 5px solid {PGA_COLORS['pumpkin']}; padding-left:15px;'><h2>2. Simulador (Métricas Acumulada)</h2></div>"))
display(widgets.VBox([drop_mes, selector_s.get_widget(), out2]))

actualizar_h(); actualizar_s()